## What an RDD actually is

An **RDD** — Resilient Distributed Dataset — is the original Spark data abstraction. Every DataFrame you write today eventually compiles down to RDD operations under the hood, so understanding RDDs explains *why* Spark behaves the way it does: lazy evaluation, fault tolerance through lineage, and partition-level parallelism.

Read the three letters of the name:

- **Resilient** — fault-tolerant. Lost partitions are recomputed from the recipe (lineage), not retrieved from a replica.
- **Distributed** — partitioned across executor memory on multiple machines.
- **Dataset** — a collection of records. A record can be any object — a string, a tuple, a custom class. Unlike a DataFrame, there is no schema and no column names.

Four more properties worth memorizing:

- **Immutable** — you never mutate an RDD. Every transformation produces a new RDD.
- **Lazy** — transformations are recorded; nothing actually runs until an action.
- **Schema-free** — there are no columns, just elements. You handle structure yourself in code.
- **Typed** — in Scala fully type-parameterized (`RDD[Int]`); in Python untyped at the language level but still typed at runtime.

## Partitions — the unit of parallelism

An RDD is split into **partitions** — chunks of data that can be processed independently. Each partition is processed by exactly one task, on one executor core. So:

- Number of partitions = degree of parallelism for that RDD.
- More partitions than cores → some tasks queue, but the cluster stays busy.
- Fewer partitions than cores → cores sit idle.

After a shuffle, the number of output partitions defaults to `spark.sql.shuffle.partitions` (200 unless you change it). For RDDs created with `parallelize`, the default count is `sc.defaultParallelism`, which is usually the total number of cores available.

![RDD Partitions](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/rdd-partitions.svg)

## Lineage — how Spark recovers from failure

Spark does **not** replicate RDD data the way HDFS does. Instead, it remembers the *recipe* — the sequence of transformations that produced each RDD. This recipe is the **lineage graph**.

If a partition is lost (executor crashes, machine dies), Spark replays just the relevant slice of the lineage to rebuild that partition. Cheap if the lineage is short, expensive if it's long.

This is also why **caching matters**. Without `.cache()`, every action re-runs the full lineage from the source. If you reuse an RDD twice, you recompute it twice.

![RDD Lineage](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/rdd-lineage.svg)

## SparkContext — the entry point for RDDs

Notebook 01 used `SparkSession`. RDDs predate SparkSession and have a different entry point: `SparkContext` (`sc` by convention). SparkSession owns one — you grab it with `spark.sparkContext`.

Mental model: `spark` is the high-level door (DataFrames, SQL); `sc` is the low-level door (RDDs, broadcast variables, accumulators). Same JVM, same cluster, just two APIs.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("RDDs")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

print("default parallelism:", sc.defaultParallelism)

## Creating RDDs

Three common ways:

- **`sc.parallelize(seq)`** — turn a Python list (already in the driver) into an RDD by distributing it across executors. Useful for tests and tiny demos.
- **`sc.textFile(path)`** — read a text file (or directory of text files) into an RDD where each element is one line.
- **`df.rdd`** — drop down from a DataFrame to its underlying RDD of `Row` objects when you need low-level control.

The bank capstone in `project/` reads real CSV/JSON/Parquet via the DataFrame API. We'll lean on `parallelize` here so the focus stays on RDD mechanics, not I/O.

In [ ]:
# A small set of card transactions in the bank-domain shape:
# (transaction_id, card_id, customer_id, merchant_category, amount, status)
txns = [
    ("T0001", "C001", "CUST001", "Groceries",      1200.00, "APPROVED"),
    ("T0002", "C002", "CUST002", "Travel",        18500.00, "APPROVED"),
    ("T0003", "C001", "CUST001", "Food",             450.00, "APPROVED"),
    ("T0004", "C003", "CUST003", "Shopping",        6700.00, "APPROVED"),
    ("T0005", "C002", "CUST002", "Fuel",            2800.00, "DECLINED"),
    ("T0006", "C001", "CUST001", "Entertainment",   1100.00, "APPROVED"),
    ("T0007", "C004", "CUST004", "Travel",        42000.00, "APPROVED"),
    ("T0008", "C003", "CUST003", "Groceries",       1850.00, "APPROVED"),
    ("T0009", "C005", "CUST005", "Shopping",        3200.00, "APPROVED"),
    ("T0010", "C002", "CUST002", "Food",              780.00, "APPROVED"),
    ("T0011", "C001", "CUST001", "Fuel",            3500.00, "APPROVED"),
    ("T0012", "C004", "CUST004", "Groceries",       2400.00, "APPROVED"),
]

txn_rdd = sc.parallelize(txns, numSlices=4)  # explicit 4 partitions

print("partitions:", txn_rdd.getNumPartitions())
print("count     :", txn_rdd.count())

## Transformations and actions — RDD edition

Same lazy model as DataFrames. Two categories:

**Transformations** build the recipe. Lazy.

| Method | Effect |
|---|---|
| `map(f)` | One in, one out. Apply `f` to each element. |
| `flatMap(f)` | One in, zero-or-more out. Flatten the result. |
| `filter(f)` | Keep elements where `f` returns truthy. |
| `distinct()` | Drop duplicates. Causes a shuffle. |
| `union(rdd2)` | Concatenate two RDDs. |
| `repartition(n)`, `coalesce(n)` | Change partition count. |

**Actions** trigger execution against the recipe.

| Method | Returns |
|---|---|
| `count()` | Number of elements |
| `take(n)` | First `n` elements |
| `first()` | First element |
| `collect()` | All elements as a Python list (driver memory) |
| `reduce(f)` | Pairwise aggregate using `f` |
| `countByValue()` | `{element: count}` dict |
| `foreach(f)` | Run `f` on each element on executors (side effects) |
| `saveAsTextFile(path)` | Write one file per partition |

In [ ]:
# Keep only APPROVED transactions, project to (merchant_category, amount).
# Two transformations — nothing has run yet.
approved = (
    txn_rdd
    .filter(lambda t: t[5] == "APPROVED")
    .map(lambda t: (t[3], t[4]))
)

print("Recipe defined. Nothing has executed yet.")
print("first 3 :", approved.take(3))   # take() is the action that triggers it

## Narrow vs wide transformations

The single most important distinction for reasoning about Spark performance.

- **Narrow transformation** — each output partition depends on **one** input partition. No data crosses the network. Examples: `map`, `filter`, `flatMap`, `mapPartitions`, `union`. These pipeline together inside a single stage.
- **Wide transformation** — each output partition depends on **many** input partitions. Data must be shuffled across the network. Examples: `reduceByKey`, `groupByKey`, `join`, `distinct`, `repartition`. Each wide transformation creates a new stage boundary.

Wide transformations are expensive — that is the rule of thumb you carry forward into the rest of the curriculum.

![Narrow vs Wide Transformations](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-narrow-wide-transforms.svg)

In [ ]:
# A narrow chain — map then filter. One stage, no shuffle.
narrow = txn_rdd.map(lambda t: (t[3], t[4])).filter(lambda kv: kv[1] > 1000)
print("--- narrow chain lineage ---")
print(narrow.toDebugString().decode())

In [ ]:
# A wide chain — reduceByKey forces a shuffle, opening a new stage.
# Look for 'ShuffledRDD' in the output — that is the shuffle boundary.
wide = txn_rdd.map(lambda t: (t[3], t[4])).reduceByKey(lambda a, b: a + b)
print("--- wide chain lineage ---")
print(wide.toDebugString().decode())

## Pair RDDs — the (key, value) view

A **Pair RDD** is just an RDD where each element is a `(key, value)` tuple. Pair RDDs unlock a family of grouping and aggregation operations that mirror SQL's `GROUP BY`.

| Method | Effect |
|---|---|
| `reduceByKey(f)` | Combine values per key using `f`. Combines locally before shuffling. |
| `groupByKey()` | Group all values per key into an iterable. Shuffles every value. |
| `mapValues(f)` | Apply `f` to values; keep keys. Narrow. |
| `flatMapValues(f)` | Apply `f`, flatten. |
| `keys()`, `values()` | Project to one side. |
| `sortByKey()` | Sort by key. Shuffles. |
| `join(rdd2)` | Inner join on key. |

To build a Pair RDD, `map` your records into `(key, value)` tuples — pick the field you want to group on as the key.

In [ ]:
# Count approved transactions per merchant category.
by_category = (
    txn_rdd
    .filter(lambda t: t[5] == "APPROVED")
    .map(lambda t: (t[3], 1))             # (merchant_category, 1)
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
)
print(by_category.collect())

## `reduceByKey` vs `groupByKey` — the exam classic

Both can compute "sum amount per merchant category." They are not equivalent.

- **`reduceByKey`** combines values **locally on each executor first** (a map-side combine), then ships the partial results across the network. Less data crosses the wire.
- **`groupByKey`** ships **every value** for every key across the network, then aggregates on the reducing side. Much more shuffle traffic. On skewed data it can OOM the executor that owns the heavy key.

The rule: if your goal is an aggregate (sum, count, max, average) and the API has a `…ByKey` form, prefer `reduceByKey` (or `aggregateByKey` / `combineByKey`) over `groupByKey`.

In [ ]:
# Sum approved transaction amounts per merchant category.
amounts_by_cat = txn_rdd.filter(lambda t: t[5] == "APPROVED").map(lambda t: (t[3], t[4]))

# Way 1: reduceByKey — map-side combine, then shuffle
sums_reduce = amounts_by_cat.reduceByKey(lambda a, b: a + b)

# Way 2: groupByKey + mapValues(sum) — ships every value across the network
sums_group = amounts_by_cat.groupByKey().mapValues(lambda values: sum(values))

print("reduceByKey :", sorted(sums_reduce.collect()))
print("groupByKey  :", sorted(sums_group.collect()))

In [ ]:
# Same answer, very different lineage. Look for the shuffle stage difference.
print("--- reduceByKey lineage ---")
print(sums_reduce.toDebugString().decode())
print()
print("--- groupByKey lineage ---")
print(sums_group.toDebugString().decode())

## Persistence — `cache()`, `persist()`, StorageLevel

Default behavior: every action re-executes the entire lineage from the source. Call `.count()` and then `.collect()` on the same RDD and you computed it twice.

`.cache()` materializes the RDD on first action and reuses it after. `.persist(level)` lets you pick the storage policy.

| StorageLevel | Where the data lives |
|---|---|
| `MEMORY_ONLY` | Executor RAM. Default for `.cache()` on RDDs. Recompute on miss. |
| `MEMORY_AND_DISK` | RAM if it fits, spill to disk otherwise. Default for `.cache()` on DataFrames. |
| `DISK_ONLY` | Always to disk. Cheap memory, slow reads. |
| `*_SER` variants | Store serialized bytes. Less RAM, more CPU. |
| `*_2` variants | Replicate each partition on two executors for resilience. |

Use `.unpersist()` when you're done — caches hold executor memory.

In [ ]:
import time

def expensive_pipeline():
    return (
        sc.parallelize(range(5_000_000), numSlices=4)
        .map(lambda x: (x, x * x))
        .filter(lambda kv: kv[1] % 7 == 0)
    )

# Without cache — every action re-runs the whole pipeline.
rdd = expensive_pipeline()
t0 = time.perf_counter()
rdd.count()
rdd.count()
print(f"no cache : two .count() calls took {time.perf_counter() - t0:.2f}s")

# With cache — first action materializes; second hits memory.
rdd_cached = expensive_pipeline().cache()
t0 = time.perf_counter()
rdd_cached.count()  # this one materializes
rdd_cached.count()  # this one hits memory
print(f"cache    : two .count() calls took {time.perf_counter() - t0:.2f}s")

rdd_cached.unpersist()

## Partition-level operations — `mapPartitions`, `repartition`, `coalesce`

Three RDD methods that operate at the partition level rather than the record level.

- **`mapPartitions(f)`** — `f` receives an iterator over one partition's records and returns an iterator. Use when per-record setup is wasteful: opening a database connection, loading a model, calling an external API in batches. One setup per partition instead of per row.
- **`repartition(n)`** — full shuffle to exactly `n` partitions. Use to *increase* parallelism or rebalance skew.
- **`coalesce(n)`** — merge to `n` partitions **without** a full shuffle. Only safe when reducing partition count. Cheap, but cannot rebalance.

Rule of thumb: shrinking → `coalesce`; growing or rebalancing → `repartition`.

In [ ]:
# mapPartitions — annotate each record with how many records share its partition
def annotate(it):
    items = list(it)
    return [(len(items), x[0]) for x in items]   # (partition_size, transaction_id)

annotated = txn_rdd.mapPartitions(annotate)
print("annotated sample :", annotated.take(4))

# repartition — full shuffle to 8 partitions
print("before repartition :", txn_rdd.getNumPartitions())
print("after  repartition :", txn_rdd.repartition(8).getNumPartitions())

# coalesce — merge down to 2 partitions, no shuffle
print("after  coalesce(2) :", txn_rdd.coalesce(2).getNumPartitions())

## Shared variables — broadcast and accumulators

Spark gives you two ways to share values across executors that are not regular RDD elements.

**Broadcast variables** ship a *read-only* value to every executor exactly once, then every task on that executor reuses it. Without broadcast, the value would be serialized into every task closure — wasteful for anything bigger than a few kilobytes. Use them for small lookup tables — a `merchant_category → spend_tier` map, an exchange-rate dict, a banned-merchant set.

**Accumulators** are *write-only-from-executors, read-only-from-driver* counters. Use them for diagnostics — counting bad records, suspicious transactions, parse failures — without breaking the lazy model. They are eventually consistent: only updates from successfully completed tasks are guaranteed.

In [ ]:
# Broadcast: a small lookup table from merchant_category to spend tier
category_tier = {
    "Groceries":     "essential",
    "Fuel":          "essential",
    "Food":          "lifestyle",
    "Entertainment": "lifestyle",
    "Shopping":      "discretionary",
    "Travel":        "discretionary",
}
b_tier = sc.broadcast(category_tier)

# Accumulator: count high-value transactions (> 10000)
high_value_count = sc.accumulator(0)

def tag(t):
    txn_id, _, _, category, amount, _ = t
    if amount > 10000:
        high_value_count.add(1)
    return (txn_id, category, b_tier.value.get(category, "unknown"), amount)

tagged = txn_rdd.map(tag)
print(tagged.take(5))
print("high-value transactions:", high_value_count.value)

b_tier.unpersist()

## RDD vs DataFrame — when to actually reach for each

|  | RDD | DataFrame |
|---|---|---|
| API style | Functional, row-by-row | Declarative, column-based |
| Optimizer | None — your code, your plan | Catalyst rewrites the plan |
| Codegen | None — Python lambdas, JVM round-trips | Tungsten generates JVM bytecode |
| Schema | None | Required |
| Performance | Slower for structured data | Faster for structured data |
| Best for | Unstructured data, custom serialization, per-partition I/O patterns | Almost everything else |

The honest guideline for the rest of this curriculum and the bank capstone in `project/`: **prefer DataFrames**. Drop to RDDs when you need something the DataFrame API genuinely cannot express — a custom `mapPartitions` pattern, a non-tabular input, or a closure that doesn't fit Spark SQL.

In [ ]:
spark.stop()